# 🎮 MuJoCo Simulation

Before putting an AI policy on a real robot, you test it in simulation.
`strands-robots` provides a MuJoCo-based physics environment that speaks the
same language as your real robot code — same observations, same actions, same policies.

**Requires:** `pip install "strands-robots[sim-mujoco]"`

## Creating a World

Everything starts with `create_simulation()` — it returns a `Simulation` object
that is both a physics engine and a Strands `AgentTool` (so AI agents can use it too).

In [ ]:
from strands_robots.simulation import create_simulation, list_backends

print(f"Available backends: {list_backends()}")
print("(mj, mjc, mjx are all aliases for mujoco)")

In [ ]:
try:
    sim = create_simulation("mujoco")
    result = sim.create_world(
        timestep=0.002,         # 500 Hz physics — standard for robot control
        gravity=[0, 0, -9.81],  # Earth gravity
        ground_plane=True,
    )
    print(result["content"][0]["text"])
    _mujoco_ok = True
except ImportError as e:
    print(f"⏭ MuJoCo not installed: {e}")
    print("  Install with: pip install 'strands-robots[sim-mujoco]'")
    _mujoco_ok = False

## Adding a Robot

Robots are loaded from the model registry. The library ships with 30+ standard
robots from MuJoCo Menagerie — just use the name.

In [ ]:
if _mujoco_ok:
    result = sim.add_robot(
        name="so100",          # This is an SO-100 robot arm
        data_config="so100",   # Links to the policy data config
        position=[0, 0, 0],
    )
    print(result["content"][0]["text"])
else:
    print('⏭ Skipped (MuJoCo not installed)')

## Building a Scene

Add objects to create a task scenario:

In [ ]:
if _mujoco_ok:
    # A table for the robot to work on
    sim.add_object("table", shape="box", position=[0.3, 0, -0.02],
                   size=[0.3, 0.3, 0.02], color=[0.6, 0.4, 0.2, 1], is_static=True)
    
    # Objects to manipulate
    sim.add_object("red_cube", shape="box", position=[0.3, 0, 0.05],
                   size=[0.03, 0.03, 0.03], color=[1, 0, 0, 1], mass=0.05)
    sim.add_object("blue_ball", shape="sphere", position=[0.2, 0.1, 0.05],
                   size=[0.02], color=[0, 0, 1, 1], mass=0.03)
    
    print("Scene built ✅")
else:
    print('⏭ Skipped (MuJoCo not installed)')

## Stepping Physics and Reading State

`step()` advances the simulation. `get_observation()` returns the same format
a real robot would — joint positions + camera images.

In [ ]:
if _mujoco_ok:
    # Let physics settle (objects falling, etc.)
    sim.step(n_steps=200)
    
    obs = sim.get_observation("so100")
    
    # Separate joint values from camera images
    joints = {k: v for k, v in obs.items() if not hasattr(v, 'shape') or getattr(v, 'ndim', 0) < 2}
    images = {k: v for k, v in obs.items() if hasattr(v, 'shape') and getattr(v, 'ndim', 0) >= 2}
    
    print("Joint positions:")
    for k, v in joints.items():
        val = f"{v:.4f}" if isinstance(v, float) else str(v)
        print(f"  {k}: {val}")
    
    print(f"\nCameras: { {k: v.shape for k, v in images.items()} }")
else:
    print('⏭ Skipped (MuJoCo not installed)')

## Rendering the Scene

Two ways to get images:
1. **`sim.render()`** → PNG bytes (for display and the AgentTool interface)
2. **`sim.get_observation()`** → raw NumPy arrays (for policies)

In [ ]:
if _mujoco_ok:
    from IPython.display import display, Image as IPImage
    
    def show_sim(sim, camera="default", width=640, height=480):
        """Render and display the simulation inline."""
        result = sim.render(camera_name=camera, width=width, height=height)
        if result.get("status") != "success":
            print(result["content"][0]["text"])
            return
        for item in result.get("content", []):
            if "image" in item:
                display(IPImage(data=item["image"]["source"]["bytes"]))
                print(result["content"][0]["text"])
                return
        print("Rendering unavailable (headless without EGL/OSMesa)")
    
    show_sim(sim)
else:
    print('⏭ Skipped (MuJoCo not installed)')

## Camera Observations

The images from `get_observation()` are raw NumPy uint8 arrays — exactly what
a policy network expects as input:

In [ ]:
obs = sim.get_observation("so100")

for k, v in obs.items():
    if hasattr(v, 'shape') and getattr(v, 'ndim', 0) >= 2:
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")

        try:
            import matplotlib.pyplot as plt
            plt.figure(figsize=(6, 4))
            plt.imshow(v)
            plt.title(f"observation['{k}']")
            plt.axis("off")
            plt.show()
        except ImportError:
            print("  (install matplotlib for inline display)")

## Sending Actions

Actions are dicts mapping joint names to target positions — same interface as
a real robot:

In [ ]:
if _mujoco_ok:
    import numpy as np
    
    # Command all joints to a sine wave position
    action = {k: float(0.3 * np.sin(1.0)) for k in joints}
    sim.send_action(action, robot_name="so100")
    
    # Step forward to let the robot move
    for _ in range(100):
        sim.step(n_steps=1)
    
    show_sim(sim)
else:
    print('⏭ Skipped (MuJoCo not installed)')

## Physics State Checkpoints

Save and restore full simulation state — useful for evaluation rollbacks:

In [ ]:
if _mujoco_ok:
    sim.save_state("before_experiment")
    
    # Do something...
    sim.step(n_steps=500)
    show_sim(sim)
    
    # Rewind
    sim.load_state("before_experiment")
    print("State restored to checkpoint ✅")
else:
    print('⏭ Skipped (MuJoCo not installed)')

In [ ]:
if _mujoco_ok:
    sim.destroy()
    print("Simulation destroyed ✅")
else:
    print('⏭ Skipped (MuJoCo not installed)')

## What's Next?

**Notebook 06** shows how to run a policy inside the simulation — the full
observe → decide → act loop, plus domain randomization and dataset recording.